In [22]:
import re
import pandas as pd
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk

In [23]:
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/prakashpun/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/prakashpun/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/prakashpun/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [24]:
df = pd.read_csv("Data.csv")

In [25]:
df.head()

,Data
0,Watch or listen live weekdays at 8:30am MT at ...
1,Watch or listen live weekdays at 8:30am MT at ...
2,"Chubby And Hot, Always Stir The Pot!"
3,"Chubby And Hot, Always Stir The Pot!"
4,"Journalist, publisher of Rebel News — telling ..."


## 1. How many records have a date that is expressed without using alphabets?

In [26]:
date_pattern = r'\d{1,2}[:]\d{1,2}'
dates_count = df['Data'].str.contains(date_pattern).sum()
print("1. Records with dates without alphabets:", dates_count)

1. Records with dates without alphabets: 133


## How many records have a word starting with the letter “w”?

In [27]:
w_pattern = r'\b[wW]\w+'
w_count = df['Data'].str.contains(w_pattern).sum()
print("2. Records with words starting with 'w':", w_count)

2. Records with words starting with 'w': 3579


## How many records make a word that starts with an alphabet and is not a URL?

In [28]:
word_pattern = r'\b[a-zA-Z]+\b(?!.*http)'
word_count = df['Data'].str.contains(word_pattern).sum()
print("3. Records with words starting with alphabet (not URL):", word_count)

3. Records with words starting with alphabet (not URL): 7305


## How many tweets contain one of these emojis :), :D, ;), :P?

In [29]:
emoji_pattern = r'[:;][)DP]'
emoji_count = df['Data'].str.contains(emoji_pattern).sum()
print("4. Records with emojis:", emoji_count)

4. Records with emojis: 18


## How many records contain a decimal number?

In [30]:
decimal_pattern = r'\d+\.\d+'
decimal_count = df['Data'].str.contains(decimal_pattern).sum()
print("5. Records with decimal numbers:", decimal_count)

5. Records with decimal numbers: 27


## What is the total number of ip addresses across all the records?

In [31]:
ip_pattern = r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b'
ip_count = len(re.findall(ip_pattern, ' '.join(df['Data'])))
print("6. Total IP addresses:", ip_count)

6. Total IP addresses: 0


## How many records have a new line?

In [32]:
newline_count = df['Data'].str.contains(r'\n').sum()
print("7. Records with new line:", newline_count)

7. Records with new line: 1211


## What is the total number of hashtags across all these tweets?

In [33]:
hashtag_pattern = r'#\w+'
hashtag_count = len(re.findall(hashtag_pattern, ' '.join(df['Data'])))
print("8. Total hashtags:", hashtag_count)

8. Total hashtags: 2924


## What is the code to substitute all non-alphanumeric characters with a new line?

In [34]:
def replace_nonalnum(text):
    return re.sub(r'[^a-zA-Z0-9]', '\n', text)

# Example usage:
print("9. Example of non-alphanumeric substitution:")
print(replace_nonalnum(df['Data'].iloc[0]))

9. Example of non-alphanumeric substitution:
Watch
or
listen
live
weekdays
at
8
30am
MT
at
ryanjespersen
com

Subscribe
via
YouTube
or
your
favourite
podcast
app


RealTalkRJ


## What is the total number of URLs across all tweets?

In [35]:
url_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
url_count = len(re.findall(url_pattern, ' '.join(df['Data'])))
print("10. Total URLs:", url_count)

10. Total URLs: 4


# B. Perform stemming and lemmatization

## Use porter stemmer to run stemming. Count the number of unique words/tokens before and after stemming

In [36]:
# Combine all text
text = ' '.join(df['Data'])
tokens = word_tokenize(text.lower())

# Count unique tokens before stemming
print("Unique tokens before stemming:", len(set(tokens)))

# Apply stemming
porter = PorterStemmer()
stemmed_words = [porter.stem(word) for word in tokens]
print("Unique tokens after stemming:", len(set(stemmed_words)))

Unique tokens before stemming: 8872
Unique tokens after stemming: 7564


## Perform lemmatization using NLTK lemmatizer . Count the number of unique words/tokens before and after lemmatization

In [37]:
# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Count unique tokens before lemmatization
print("Unique tokens before lemmatization:", len(set(tokens)))

# Apply lemmatization
lemmatized_words = [lemmatizer.lemmatize(word) for word in tokens]
print("Unique tokens after lemmatization:", len(set(lemmatized_words)))

Unique tokens before lemmatization: 8872
Unique tokens after lemmatization: 8338


## Compare the change in word frequencies from stemming and lemmatization. Which are the top 10 words after stemming/lemmatization

In [38]:
# Get frequencies
from collections import Counter

# Stemming frequencies
stem_freq = Counter(stemmed_words)
print("\nTop 10 words after stemming:")
print(pd.Series(stem_freq).sort_values(ascending=False).head(10))

# Lemmatization frequencies
lem_freq = Counter(lemmatized_words)
print("\nTop 10 words after lemmatization:")
print(pd.Series(lem_freq).sort_values(ascending=False).head(10))


Top 10 words after stemming:
.      10065
,       9314
and     3514
#       2936
the     2859
of      2223
to      2179
&       1699
@       1620
a       1551
dtype: int64

Top 10 words after lemmatization:
.      10065
,       9314
and     3514
#       2936
the     2859
of      2223
to      2179
a       1768
&       1699
@       1620
dtype: int64


## What is the change in word frequencies if normalization is done after stop word removal.

In [39]:
# Remove stop words
stop_words = set(stopwords.words('english'))
filtered_tokens = [word for word in tokens if word.lower() not in stop_words]

# Get frequencies after stop word removal
filtered_freq = Counter(filtered_tokens)
print("\nWord frequencies after stop word removal:")
print(pd.Series(filtered_freq).sort_values(ascending=False).head(10))


Word frequencies after stop word removal:
.          10065
,           9314
#           2936
&           1699
@           1620
|           1231
alberta     1059
!            972
’            660
:            625
dtype: int64
